# Plant Disease Classification using ResNet50V2

This notebook adapts the training pipeline in the supplied MobileNetV2 and EfficientNetB0 notebooks for **ResNet50V2**.

- Dataset: `vipoooool/new-plant-diseases-dataset`
- Expected classes: 38
- Input size: 224 × 224
- ImageNet pretrained weights
- Two-phase transfer learning: classifier-head training, then fine-tuning
- `tf.data` prefetch, GPU mixed precision when available
- Label smoothing, early stopping, best-model checkpoint and learning-rate reduction
- Full validation evaluation, confusion matrix, per-class F1, model size and latency

**Before running:** In Google Colab, select **Runtime → Change runtime type → T4 GPU**. The validation cache may require several GB of RAM. If memory is insufficient, use the uncached alternative noted in the input-pipeline cell.

## 1. Setup and configuration

In [ ]:
import os, json, time, random, glob, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# ---------------- Configuration ----------------
MODEL_NAME       = "resnet50v2"
IMG_SIZE         = (224, 224)
BATCH_SIZE       = 64
WARMUP_EPOCHS    = 3
FINETUNE_EPOCHS  = 8
FINETUNE_LAYERS  = 60
WEIGHTS          = "imagenet"
OUT              = "/content"
SEED             = 42
# ------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE (enable a GPU runtime for faster training)")
if gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

for folder in ["models", "results", "figures"]:
    os.makedirs(f"{OUT}/{folder}", exist_ok=True)

print("TensorFlow:", tf.__version__)

## 2. Download the dataset

In [ ]:
try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

try:
    DATA_ROOT = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
except Exception as exc:
    print("Dataset download failed:", str(exc)[:200])
    print("Complete Kaggle authentication if prompted, then rerun this cell.")
    kagglehub.login()
    DATA_ROOT = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")

print("Dataset path:", DATA_ROOT)

## 3. Locate train and validation folders

In [ ]:
def find_split_dirs(root):
    train_dir = None
    valid_dir = None

    for dirpath, dirnames, _ in os.walk(root):
        base = os.path.basename(dirpath).lower()
        if len(dirnames) == 38:
            if base == "train":
                train_dir = dirpath
            elif base in ("valid", "validation"):
                valid_dir = dirpath

    return train_dir, valid_dir

TRAIN_DIR, VALID_DIR = find_split_dirs(DATA_ROOT)

assert TRAIN_DIR is not None, "Could not find the training folder with 38 class folders."
assert VALID_DIR is not None, "Could not find the validation folder with 38 class folders."

print("Train:", TRAIN_DIR)
print("Valid:", VALID_DIR)

## 4. Input pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

valid_raw = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_raw.class_names
assert class_names == valid_raw.class_names, "Train/validation class order differs."
NUM_CLASSES = len(class_names)
valid_paths = valid_raw.file_paths

print(
    "Classes:", NUM_CLASSES,
    "| train images:", len(train_raw.file_paths),
    "| validation images:", len(valid_paths)
)
assert NUM_CLASSES == 38, f"Expected 38 classes, found {NUM_CLASSES}"

opts = tf.data.Options()
opts.deterministic = False
train_ds = train_raw.with_options(opts).prefetch(AUTOTUNE)

# Cache validation images in RAM, following the supplied notebook pipeline.
# If Colab runs out of RAM, replace the block below with:
# valid_ds = valid_raw.prefetch(AUTOTUNE)
valid_ds = (
    valid_raw
    .map(lambda x, y: (tf.cast(x, tf.uint8), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)

## 5. Build the ResNet50V2 model

In [ ]:
@tf.keras.utils.register_keras_serializable()
def resnet50v2_preprocess(images):
    # ResNet50V2 preprocessing converts RGB pixels to the model's expected range.
    return tf.keras.applications.resnet_v2.preprocess_input(images)

def build_model(num_classes):
    backbone = tf.keras.applications.ResNet50V2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights=WEIGHTS
    )
    backbone.trainable = False

    augment = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name="augment")

    inputs = Input(shape=IMG_SIZE + (3,), name="image")
    x = augment(inputs)
    x = layers.Lambda(
        resnet50v2_preprocess,
        name="resnet50v2_preprocessing"
    )(x)
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions"
    )(x)

    return Model(inputs, outputs, name=MODEL_NAME), backbone

def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )

model, backbone = build_model(NUM_CLASSES)
model_path = f"{OUT}/models/{MODEL_NAME}.keras"
print("Total parameters: %.2fM" % (model.count_params() / 1e6))

## 6. Phase 1: train the classifier head (backbone frozen)

In [ ]:
t_start = time.time()

compile_model(model, 1e-3)
h1 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=WARMUP_EPOCHS,
    verbose=1
)

## 7. Phase 2: fine-tune the top backbone layers

In [ ]:
backbone.trainable = True

for layer in backbone.layers[:-FINETUNE_LAYERS]:
    layer.trainable = False

# Keep BatchNormalization layers frozen for stable fine-tuning.
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

compile_model(model, 1e-4)

callbacks = [
    EarlyStopping(
        monitor="val_accuracy",
        patience=2,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        model_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=1,
        min_lr=1e-6
    )
]

h2 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=WARMUP_EPOCHS + FINETUNE_EPOCHS,
    initial_epoch=WARMUP_EPOCHS,
    callbacks=callbacks,
    verbose=1
)

train_minutes = (time.time() - t_start) / 60
print("Total training time: %.1f min" % train_minutes)

keys = ["loss", "accuracy", "val_loss", "val_accuracy"]
history = {k: h1.history[k] + h2.history[k] for k in keys}
pd.DataFrame(history).to_csv(
    f"{OUT}/results/{MODEL_NAME}_history.csv",
    index_label="epoch"
)

## 8. Training curves

In [ ]:
epochs = np.arange(1, len(history["loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, key, title in [
    (axes[0], "loss", "Loss"),
    (axes[1], "accuracy", "Accuracy")
]:
    ax.plot(epochs, history[key], marker="o", linewidth=2, label="Train")
    ax.plot(epochs, history["val_" + key], marker="s", linewidth=2, label="Validation")
    ax.axvline(WARMUP_EPOCHS + 0.5, color="gray", linestyle="--",
               label="Fine-tuning starts")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.savefig(
    f"{OUT}/figures/{MODEL_NAME}_training_curves.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

## 9. Evaluation on the full validation set

In [ ]:
best = tf.keras.models.load_model(model_path)

probs = best.predict(valid_ds, verbose=1)
y_pred = probs.argmax(axis=1)
confidence = probs.max(axis=1)

y_true = np.array([
    class_names.index(os.path.basename(os.path.dirname(path)))
    for path in valid_paths
])

metrics = {
    "Accuracy": accuracy_score(y_true, y_pred),
    "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
    "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    "Weighted Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "Weighted Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
}

metrics_df = pd.DataFrame(metrics.items(), columns=["Metric", "Score"])
metrics_df.to_csv(
    f"{OUT}/results/{MODEL_NAME}_overall_metrics.csv",
    index=False
)
display(metrics_df.style.format({"Score": "{:.2%}"}))

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(
    f"{OUT}/results/{MODEL_NAME}_classification_report.csv"
)

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(20, 18))
sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix ({MODEL_NAME})")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(
    f"{OUT}/figures/{MODEL_NAME}_confusion_matrix.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

## 10. Per-class F1 and difficult classes

In [ ]:
per_class = report_df.loc[class_names].sort_values("f1-score")

plt.figure(figsize=(12, 10))
plt.barh(per_class.index, per_class["f1-score"])
plt.xlabel("F1-score")
plt.title("Per-Class F1 Scores")
plt.tight_layout()
plt.savefig(
    f"{OUT}/figures/{MODEL_NAME}_per_class_f1.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

print("Lowest 5 classes:")
display(per_class.head(5)[["precision", "recall", "f1-score", "support"]])

print("Highest 5 classes:")
display(per_class.tail(5)[["precision", "recall", "f1-score", "support"]])

## 11. Efficiency: parameters, model size and inference latency

In [ ]:
sample = (np.random.rand(1, *IMG_SIZE, 3) * 255).astype("float32")

for _ in range(10):
    best(sample, training=False).numpy()

n = 100
t0 = time.perf_counter()
for _ in range(n):
    best(sample, training=False).numpy()

latency_ms = (time.perf_counter() - t0) / n * 1000
device = tf.config.list_physical_devices("GPU")

efficiency = {
    "model": MODEL_NAME,
    "accuracy": float(metrics["Accuracy"]),
    "macro_f1": float(metrics["Macro F1"]),
    "weighted_f1": float(metrics["Weighted F1"]),
    "params_m": float(best.count_params() / 1e6),
    "size_mb": float(os.path.getsize(model_path) / 1e6),
    "latency_ms": float(latency_ms),
    "train_minutes": float(train_minutes),
    "device": "GPU" if device else "CPU",
}

with open(f"{OUT}/results/{MODEL_NAME}_metrics.json", "w") as f:
    json.dump(efficiency, f, indent=2)

display(pd.DataFrame([efficiency]))

## 12. Compare metrics JSON files available in this Colab session

In [ ]:
rows = [
    json.load(open(path))
    for path in sorted(glob.glob(f"{OUT}/results/*_metrics.json"))
]

if rows:
    comparison = pd.DataFrame(rows)
    comparison.to_csv(
        f"{OUT}/results/model_comparison.csv",
        index=False
    )
    display(comparison.style.format({
        "accuracy": "{:.2%}",
        "macro_f1": "{:.2%}",
        "weighted_f1": "{:.2%}",
        "params_m": "{:.2f}",
        "size_mb": "{:.1f}",
        "latency_ms": "{:.1f}",
        "train_minutes": "{:.1f}"
    }))
else:
    print("No model metrics JSON files found.")

## 13. Predict on one uploaded leaf image

In [ ]:
from google.colab import files as colab_files

uploaded = colab_files.upload()

for filename, content in uploaded.items():
    image_path = f"{OUT}/{filename}"
    with open(image_path, "wb") as f:
        f.write(content)

    image = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(image)
    prediction = best.predict(np.expand_dims(arr, 0), verbose=0)[0]

    top_indices = prediction.argsort()[-5:][::-1]
    print(f"Top predictions for {filename}:")
    for idx in top_indices:
        print(f"{class_names[idx]}: {prediction[idx]:.4f}")

## 14. Download the trained model

Run this cell to download the trained ResNet50V2 model. The CSV/JSON results and plots are saved under `/content/results` and `/content/figures` in Colab.

In [ ]:
from google.colab import files
files.download(model_path)